# G12- test run for XGBoost on FashionMNIST dataset

In [1]:
# import relevant libraries
import torch
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import time
torch.manual_seed(42)
# XGBoost Specific libraries
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score, classification_report, roc_auc_score
from xgboost import XGBClassifier

In [2]:
# check for GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu") # will not need GPU for XGBoost
print("Device:", device)
# verify Pytorch and CUDA versions
cuda_version = torch.version.cuda
print(f"cuda version{cuda_version}")
print(f"PyTorch version: {torch.__version__}")

Device: cuda
cuda version13.0
PyTorch version: 2.9.0+cu130


### Get the FashionMNIST dataset

In [3]:
# 1.Load the Fashion-MNIST data

# set seed
SEED=42
torch.manual_seed(42)
print("\n1. LOADING FASHION-MNIST DATA")
transform = transforms.Compose([transforms.ToTensor()])

# download the training and test data
train_dataset = datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform) # full train 
test_dataset = datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform) # full test

# print to see the length of the datasets
print("\n===DIMENSIONS OF THE FASHION-MNIST DATA===")
print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")


1. LOADING FASHION-MNIST DATA

===DIMENSIONS OF THE FASHION-MNIST DATA===
Training samples: 60000
Test samples: 10000


In [4]:
# convert the dataset from tensor to numpy, flatten and normalize
X_train_reshaped = train_dataset.data.numpy().reshape(-1, 28*28) /255
y_train_reshaped = train_dataset.targets.numpy()

# convert the dataset from tensor to numpy, flatten and normalize
X_test= test_dataset.data.numpy().reshape(-1, 28*28) /255
y_test= test_dataset.targets.numpy()

print("Train size:", X_train_reshaped.shape)
print("Test size:", X_test.shape)

Train size: (60000, 784)
Test size: (10000, 784)


In [5]:
# split the training dataset using train_test_split and stratified sampling
X_train, X_val, y_train, y_val = train_test_split(
    X_train_reshaped,
    y_train_reshaped,
    test_size=10000, # this will split the train set into 50k-train and 10-val
    stratify=y_train_reshaped,
    random_state=SEED
)

print("Training set:", X_train.shape)
print("Validation set:", X_val.shape)
print ("Test set:", X_test.shape)

Training set: (50000, 784)
Validation set: (10000, 784)
Test set: (10000, 784)


In [8]:
# get the mean and std of the data. May be beeded for MLP and CNN 
print(X_train.mean())
print(X_train.std())

0.28627582833133247
0.3531853296234762


These are the global mean for the entire FashionMNIST dataset. This would be handy when we to normalize for the MLP and CNN 

In [9]:
# return the text labels
#def text_labels(self, indices):
#    """Return text labels."""
#    labels_indices = ['t-shirt', 'trouser', 'pullover', 'dress', 'coat',
#              'sandal', 'shirt', 'sneaker', 'bag', 'ankle boot']
#    return [labels[int(i)] for i in label_indices]

#instance(data, fashion_mnist)

#for images, labels in train_loader:
#    print('Image batch dimensions:', images.shape)
#    print('Image label dimensions:', labels.shape)
#    break

In [10]:
# train the base model
xgb_base_model = XGBClassifier(
    objective="multi:softprob",
    num_class=10,
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=42   
)


In [ ]:
print("Please grab a drink, G12 Team is training their XGBoost...")
xgb_base_model.fit(X_train, y_train)

Please grab a drink, G12 Team is training their XGBoost...


In [ ]:
# get some metrics
xgb_val_preds = xgb_base_model.predict(X_val)
xgb_val_acc = accuracy_score(y_val, xgb_val_preds)
xgb_val_auc= roc_auc_score(y_val, xgb_val_preds)

print("\nValidation Accuracy:", xgb_val_acc)
print("\nValidation Classification Report:")
print(classification_report(y_val, xgb_val_preds))

In [ ]:
# calculate accuracy and AUC
xgb_test_acc=accuracy_score(y_test, pred_labels_np)
xgb_test_auc=roc_auc_score(y_test, pred_probs_np)


In [ ]:
# Build the fine tuned model
xgb_tuned_1=XGBClassifier(
    objective="multi:softprob",
    max_depth=3,
    n_estimators=100,
    learning_rate=0.01
)

In [ ]:
print("Team G12 first tuned XGBoost...")
xgb_tuned_1.fit(X_train, y_train)

In [ ]:
#hyperparameters = {'learning_rate': 0.27741698265128945,
#                   'max_depth': 10,
#                   'min_child_weight': 2,
#                   'n_estimators': 116,
#                   'subsample': 0.8251291379552849,
#                   'seed': SEED,
#                   'objective' : 'multi:softprob',
#                   'num_class':10
#                   }

# Initialize the XGBoost classifier with best parameters
#xgb_best_model = xgb.XGBClassifier(**hyperparameters)

# Fit the model on the entire training set
#xgb_best_model.fit(X_train, y_train)